# Phase 2B: Item-Based Collaborative Filtering

In this notebook, we build an Item-Based Collaborative Filtering (CF) recommendation model. Instead of finding similar users, Item-CF finds similar movies based on how users rated them. It is computationally efficient and provides easy-to-understand recommendations.

We will:
1. Load the modeling subset.
2. Build a sparse User-Item interaction matrix.
3. Compute an Item-Item cosine similarity matrix.
4. Develop functions to find similar movies and generate personalized recommendations for users.


In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')


## 1. Load Data
We load the `netflix_modeling_subset.csv` and the raw `movie_titles.csv` to map movie IDs to actual titles.


In [2]:
# Load the interaction subset
df = pd.read_csv('../data/processed/netflix_modeling_subset.csv')

# Load movie titles for interpretation
titles_df = pd.read_csv('../data/raw/movie_titles.csv', encoding='ISO-8859-1', header=None, names=['movie_id', 'year', 'title'], on_bad_lines='skip')
title_map = dict(zip(titles_df['movie_id'], titles_df['title']))

print(f"Loaded {len(df)} ratings.")


Loaded 43219416 ratings.


## 2. Create User-Item Interaction Matrix
We convert the tabular dataframe into a sparse matrix. A sparse matrix only stores non-zero values, saving significant memory.


In [3]:
# Create categorical types to assign contiguous indices
user_ids = df['user_id'].astype('category')
movie_ids = df['movie_id'].astype('category')

user_idx = user_ids.cat.codes
movie_idx = movie_ids.cat.codes

# Create mapping dictionaries to translate indices back to IDs
user_map = dict(enumerate(user_ids.cat.categories))
movie_map = dict(enumerate(movie_ids.cat.categories))

rev_user_map = {v: k for k, v in user_map.items()}
rev_movie_map = {v: k for k, v in movie_map.items()}

# Construct the sparse matrix
sparse_user_item = csr_matrix((df['rating'], (user_idx, movie_idx)), shape=(len(user_map), len(movie_map)))

print(f"Sparse matrix shape: {sparse_user_item.shape}")
print(f"Number of non-zero elements: {sparse_user_item.nnz}")


Sparse matrix shape: (50000, 7291)
Number of non-zero elements: 43219416


## 3. Compute Item-Item Similarity Matrix
We calculate the cosine similarity between every pair of movies using their rating vectors.


In [4]:
# Transpose so movies are rows and users are columns
sparse_item_user = sparse_user_item.T

# Calculate cosine similarity.
item_similarity = cosine_similarity(sparse_item_user)

# Set diagonal to zero to avoid recommending a movie to itself
np.fill_diagonal(item_similarity, 0)

print(f"Item similarity matrix shape: {item_similarity.shape}")


Item similarity matrix shape: (7291, 7291)


## 4. Item-to-Item Recommendations
This function takes a single movie and returns the most similar movies based on our computed cosine similarities.


In [5]:
def recommend_movies(movie_id, top_n=10):
    if movie_id not in rev_movie_map:
        return pd.DataFrame()
    
    idx = rev_movie_map[movie_id]
    sim_scores = item_similarity[idx]
    
    # Sort indices in descending order and get top_n
    top_indices = np.argsort(sim_scores)[::-1][:top_n]
    
    results = []
    for i in top_indices:
        m_id = movie_map[i]
        title = title_map.get(m_id, f"Unknown (ID: {m_id})")
        results.append({"movie_id": m_id, "title": title, "similarity": round(sim_scores[i], 4)})
        
    return pd.DataFrame(results)

# Let's test with 5 recognizable movies
test_movies = [1, 28, 30, 26, 33]

for m in test_movies:
    if m in rev_movie_map:
        title = title_map.get(m, f"ID: {m}")
        print(f"--- Movies similar to: {title} ---")
        display(recommend_movies(m, top_n=5))
        print()


--- Movies similar to: Lilo and Stitch ---


,movie_id,title,similarity
0,8743,Ice Age,0.6626
1,11089,Unknown (ID: 11089),0.6526
2,6510,A Bug's Life,0.6427
3,2690,The Emperor's New Groove,0.6358
4,3962,Finding Nemo (Widescreen),0.6349



--- Movies similar to: Something's Gotta Give ---


,movie_id,title,similarity
0,6206,My Big Fat Greek Wedding,0.7805
1,6287,Pretty Woman,0.7770
2,11283,Forrest Gump,0.7739
3,4656,Erin Brockovich,0.7649
4,1905,Pirates of the Caribbean: The Curse of the Bla...,0.7621



--- Movies similar to: Never Die Alone ---


,movie_id,title,similarity
0,11910,Torque,0.3453
1,15712,Cradle 2 the Grave,0.3343
2,9856,My Baby's Daddy,0.3272
3,17361,Johnson Family Vacation,0.3186
4,17395,Breakin' All the Rules,0.3155



--- Movies similar to: Aqua Teen Hunger Force: Vol. 1 ---


,movie_id,title,similarity
0,5297,Aqua Teen Hunger Force: Vol. 2,0.7929
1,15773,Aqua Teen Hunger Force: Vol. 3,0.7690
2,7496,Sealab 2021: Season 1,0.5675
3,17230,Sealab 2021: Season 2,0.5285
4,12834,Family Guy: Vol. 2: Season 3,0.4169


## 5. Personalized Recommendations
To recommend movies for a user, we take their existing ratings and compute a weighted score for unrated movies using the item-item similarity matrix.


In [6]:
def recommend_for_user(user_id, top_n=10):
    if user_id not in rev_user_map:
        return pd.DataFrame()
    
    idx = rev_user_map[user_id]
    user_ratings = sparse_user_item[idx].toarray().flatten()
    
    # Predict scores by taking the dot product of item similarities and the user's rating vector
    scores = item_similarity.dot(user_ratings)
    
    # Exclude movies the user has already rated
    scores[user_ratings > 0] = -1 
    
    top_indices = np.argsort(scores)[::-1][:top_n]
    
    results = []
    for i in top_indices:
        m_id = movie_map[i]
        title = title_map.get(m_id, f"Unknown (ID: {m_id})")
        results.append({"movie_id": m_id, "title": title, "predicted_score": round(scores[i], 2)})
        
    return pd.DataFrame(results)

# Test with 5 users from the dataset
sample_users = df['user_id'].unique()[:5]

for u in sample_users:
    print(f"--- Top 5 Recommendations for User: {u} ---")
    display(recommend_for_user(u, top_n=5))
    print()


--- Top 5 Recommendations for User: 712664 ---


,movie_id,title,predicted_score
0,11283,Forrest Gump,2506.61
1,3962,Finding Nemo (Widescreen),2358.89
2,14621,Shrek (Full-screen),2353.84
3,11490,A League of Their Own,2286.53
4,3938,Shrek 2,2268.97



--- Top 5 Recommendations for User: 1331154 ---


,movie_id,title,predicted_score
0,3860,Bruce Almighty,1884.37
1,11089,Unknown (ID: 11089),1871.82
2,6408,Unknown (ID: 6408),1802.44
3,15062,Grease,1799.26
4,1962,50 First Dates,1779.18



--- Top 5 Recommendations for User: 439011 ---


,movie_id,title,predicted_score
0,10042,Raiders of the Lost Ark,949.86
1,14550,The Shawshank Redemption: Special Edition,943.21
2,2782,Braveheart,937.24
3,16954,Indiana Jones and the Last Crusade,936.34
4,14670,Batman,914.29



--- Top 5 Recommendations for User: 1644750 ---


,movie_id,title,predicted_score
0,14550,The Shawshank Redemption: Special Edition,1500.24
1,14410,Spider-Man,1495.76
2,11064,Pulp Fiction,1473.54
3,17157,Saving Private Ryan,1454.07
4,9960,Die Hard,1432.72



--- Top 5 Recommendations for User: 2031561 ---


,movie_id,title,predicted_score
0,11283,Forrest Gump,1029.71
1,10042,Raiders of the Lost Ark,1027.28
2,8387,Minority Report,1018.59
3,8904,Good Will Hunting,1012.20
4,8644,Catch Me If You Can,1002.60


## 6. Discussion

**Strengths of Item-Based CF:**
- **Explainability:** It is very easy to explain recommendations to users (e.g., "We recommend Movie B because you gave a high rating to Movie A").
- **Stability:** Movie similarities tend to be more stable over time than user tastes, meaning the similarity matrix does not need to be updated as frequently as user profiles.

**Weaknesses of Item-Based CF:**
- **Popularity Bias:** Highly rated blockbuster movies tend to be similar to everything, meaning obscure or niche movies are rarely recommended.
- **Sparsity Challenges:** If two niche movies have very few overlapping users, their cosine similarity will be near zero, making them hard to recommend together.

**Expected Cold-Start Issues:**
- **New Users:** If a user registers and has 0 ratings, the rating vector is entirely zeros. The dot product yields a score of 0 for all items, resulting in no recommendations.
- **New Movies:** If a movie is added to the catalog and has no ratings, its similarity to all other items is 0. It won't appear in recommendations until initial ratings are gathered.
